# Method 6: Agarwal's Model MCCP Calibration

This notebook builds Mondrian Cross Conformal Prediction calibration data for the **Agarwal's** model using only the prepared `train` and `val` splits from `classification_data`.

### Workflow
1. Load `X_train / y_train` and `X_val / y_val`.
2. Perform `10`-fold random shuffled CV on `X_train / y_train`.
3. For each fold, train the bias-initialized classifier on the fold-fit split.
4. Use the separate validation split `X_val / y_val` to choose the fold-best epoch by **validation loss only**.
5. Score the held-out fold with that fold-best model and save the held-out predictions.
6. Concatenate all held-out predictions into a calibration CSV and export the MCCP `qhat` CSV with `MCPTool`.


In [1]:
from pathlib import Path
import copy
import random

import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import KFold
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from mcp_tool import MCPTool

sns.set_theme(style="darkgrid")

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


Using device: cpu


In [2]:
RANDOM_STATE = 0
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [3]:
DATA_DIR = Path("./classification_data")
OUTPUT_DIR = Path("./agarwal_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = OUTPUT_DIR / "mccp_fold_models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

HOLDOUT_DIR = OUTPUT_DIR / "mccp_heldout_folds"
HOLDOUT_DIR.mkdir(parents=True, exist_ok=True)

CALIBRATION_CSV = OUTPUT_DIR / "agarwal_calibration_mccp.csv"
QHAT_CSV = OUTPUT_DIR / "agarwal_qhat_mccp.csv"
FOLD_SUMMARY_CSV = OUTPUT_DIR / "agarwal_mccp_fold_summary.csv"

BATCH_SIZE = 2048
LEARNING_RATE = 1e-3
EPOCHS = 2500
CV_FOLDS = 5
MCP_ALPHA = 0.1

X_train = np.load(DATA_DIR / "X_train.npy").astype(np.float32)
X_val = np.load(DATA_DIR / "X_val.npy").astype(np.float32)
y_train = np.load(DATA_DIR / "y_train.npy").astype(np.int64)
y_val = np.load(DATA_DIR / "y_val.npy").astype(np.int64)

print("train:", X_train.shape, y_train.shape)
print("val:", X_val.shape, y_val.shape)
print("output_dir:", OUTPUT_DIR.resolve())


train: (1604, 7) (1604,)
val: (200, 7) (200,)
output_dir: /home/srinadb/BCU/BCUTorch/agarwal_outputs


In [4]:
def make_prediction_dataframe(fsrq_probabilities, targets):
    fsrq_probabilities = np.asarray(fsrq_probabilities, dtype=np.float64).reshape(-1)
    fsrq_probabilities = np.clip(fsrq_probabilities, 1e-7, 1.0 - 1e-7)
    targets = np.asarray(targets, dtype=np.int64).reshape(-1)
    true_labels = np.where(targets == 1, "FSRQ", "BLL")
    return pd.DataFrame(
        {
            "true_label": true_labels,
            "BLL": 1.0 - fsrq_probabilities,
            "FSRQ": fsrq_probabilities,
        }
    )


def make_binary_metrics(y_true, probabilities):
    y_true = np.asarray(y_true).astype(int)
    probabilities = np.asarray(probabilities, dtype=np.float64).reshape(-1)
    predictions = (probabilities >= 0.5).astype(int)

    metrics = {
        "accuracy": accuracy_score(y_true, predictions),
        "precision_weighted": precision_score(y_true, predictions, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, predictions, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true, predictions, average="weighted", zero_division=0),
    }

    try:
        metrics["auc"] = roc_auc_score(y_true, probabilities)
    except ValueError:
        metrics["auc"] = np.nan

    return metrics


def make_loaders(X_fit, y_fit, X_valid, y_valid):
    train_dataset = TensorDataset(
        torch.tensor(X_fit, dtype=torch.float32),
        torch.tensor(y_fit, dtype=torch.float32).unsqueeze(1),
    )
    valid_dataset = TensorDataset(
        torch.tensor(X_valid, dtype=torch.float32),
        torch.tensor(y_valid, dtype=torch.float32).unsqueeze(1),
    )
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return train_loader, valid_loader


def collect_probabilities(model, X):
    model.eval()
    with torch.no_grad():
        probabilities = model(torch.tensor(X, dtype=torch.float32, device=device))
    return probabilities.detach().cpu().numpy().reshape(-1)


def build_agarwal_model(input_dim):
    return nn.Sequential(
        nn.Linear(input_dim, 64),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 1),
        nn.Sigmoid(),
    ).to(device)


def train_agarwal_model(model, train_loader, valid_loader, epochs):
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    history_rows = []
    best_state = copy.deepcopy(model.state_dict())
    best_epoch = 1
    best_loss = np.inf
    best_metrics = None

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_probabilities = []
        train_targets = []

        for features_batch, targets_batch in train_loader:
            features_batch = features_batch.to(device)
            targets_batch = targets_batch.to(device)

            optimizer.zero_grad(set_to_none=True)
            probabilities = model(features_batch)
            loss = criterion(probabilities, targets_batch)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * targets_batch.size(0)
            train_probabilities.append(probabilities.detach().cpu().numpy())
            train_targets.append(targets_batch.detach().cpu().numpy())

        train_probabilities = np.concatenate(train_probabilities, axis=0).reshape(-1)
        train_targets = np.concatenate(train_targets, axis=0).reshape(-1).astype(int)
        train_loss = train_loss_sum / len(train_loader.dataset)
        train_metrics = make_binary_metrics(train_targets, train_probabilities)

        model.eval()
        valid_loss_sum = 0.0
        valid_probabilities = []
        valid_targets = []

        with torch.no_grad():
            for features_batch, targets_batch in valid_loader:
                features_batch = features_batch.to(device)
                targets_batch = targets_batch.to(device)

                probabilities = model(features_batch)
                loss = criterion(probabilities, targets_batch)

                valid_loss_sum += loss.item() * targets_batch.size(0)
                valid_probabilities.append(probabilities.detach().cpu().numpy())
                valid_targets.append(targets_batch.detach().cpu().numpy())

        valid_probabilities = np.concatenate(valid_probabilities, axis=0).reshape(-1)
        valid_targets = np.concatenate(valid_targets, axis=0).reshape(-1).astype(int)
        valid_loss = valid_loss_sum / len(valid_loader.dataset)
        valid_metrics = make_binary_metrics(valid_targets, valid_probabilities)

        history_rows.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "valid_loss": valid_loss,
                "train_accuracy": train_metrics["accuracy"],
                "valid_accuracy": valid_metrics["accuracy"],
                "valid_auc": valid_metrics["auc"],
                "valid_f1_weighted": valid_metrics["f1_weighted"],
            }
        )

        if valid_loss < best_loss:
            best_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            best_metrics = {
                "valid_loss": valid_loss,
                **valid_metrics,
            }

    history_df = pd.DataFrame(history_rows)
    last_state = copy.deepcopy(model.state_dict())
    return history_df, last_state, best_state, best_epoch, best_metrics


In [5]:
kfold = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

calibration_frames = []
fold_summaries = []

for fold_index, (fit_index, holdout_index) in enumerate(kfold.split(X_train), start=1):
    print(f"Fold {fold_index}/{CV_FOLDS}")
    fold_dir = MODEL_DIR / f"fold_{fold_index:02d}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    holdout_csv = HOLDOUT_DIR / f"agarwal_holdout_fold_{fold_index:02d}_mccp.csv"

    X_fit = X_train[fit_index]
    y_fit = y_train[fit_index]
    X_holdout = X_train[holdout_index]
    y_holdout = y_train[holdout_index]

    fit_bll = int((y_fit == 0).sum())
    fit_fsrq = int((y_fit == 1).sum())
    holdout_bll = int((y_holdout == 0).sum())
    holdout_fsrq = int((y_holdout == 1).sum())

    print(f"  Fit count: {len(fit_index)} | BLL: {fit_bll} | FSRQ: {fit_fsrq}")
    print(f"  Held-out count: {len(holdout_index)} | BLL: {holdout_bll} | FSRQ: {holdout_fsrq}")

    train_loader, valid_loader = make_loaders(X_fit, y_fit, X_val, y_val)

    model = build_agarwal_model(X_train.shape[1])
    history_df, last_state, best_state, best_epoch, best_metrics = train_agarwal_model(
        model,
        train_loader,
        valid_loader,
        EPOCHS,
    )
    history_df.to_csv(fold_dir / "history.csv", index=False)
    torch.save({"model_state_dict": last_state}, fold_dir / "final_last.pt")
    torch.save(
        {
            "model_state_dict": best_state,
            "best_epoch": best_epoch,
            "best_metrics": best_metrics,
        },
        fold_dir / "final_best_by_val_loss.pt",
    )

    model.load_state_dict(best_state)
    holdout_probabilities = collect_probabilities(model, X_holdout)
    holdout_df = make_prediction_dataframe(holdout_probabilities, y_holdout)
    holdout_df.to_csv(holdout_csv, index=False)
    calibration_frames.append(holdout_df)

    holdout_metrics = make_binary_metrics(y_holdout, holdout_probabilities)
    fold_summaries.append(
        {
            "fold": fold_index,
            "fit_samples": len(fit_index),
            "heldout_samples": len(holdout_index),
            "fit_bll": fit_bll,
            "fit_fsrq": fit_fsrq,
            "heldout_bll": holdout_bll,
            "heldout_fsrq": holdout_fsrq,
            "best_epoch_by_val_loss": best_epoch,
            "best_val_loss": best_metrics["valid_loss"],
            "best_val_accuracy": best_metrics["accuracy"],
            "best_val_auc": best_metrics["auc"],
            "best_val_f1_weighted": best_metrics["f1_weighted"],
            "heldout_accuracy": holdout_metrics["accuracy"],
            "heldout_auc": holdout_metrics["auc"],
            "heldout_f1_weighted": holdout_metrics["f1_weighted"],
            "heldout_csv": str(holdout_csv),
            "best_model_path": str(fold_dir / "final_best_by_val_loss.pt"),
        }
    )

calibration_df = pd.concat(calibration_frames, ignore_index=True)
calibration_df.to_csv(CALIBRATION_CSV, index=False)

fold_summary_df = pd.DataFrame(fold_summaries)
fold_summary_df.to_csv(FOLD_SUMMARY_CSV, index=False)

calibration_targets = np.where(calibration_df["true_label"].to_numpy() == "FSRQ", 1, 0)
calibration_probabilities = calibration_df["FSRQ"].to_numpy(dtype=np.float64)
calibration_metrics = make_binary_metrics(calibration_targets, calibration_probabilities)

print("Saved calibration CSV:", CALIBRATION_CSV)
print("Saved fold summary CSV:", FOLD_SUMMARY_CSV)
print()
print("Combined calibration metrics:")
print(pd.Series(calibration_metrics))
print()

fold_summary_df


Fold 1/5
  Fit count: 1283 | BLL: 838 | FSRQ: 445
  Held-out count: 321 | BLL: 215 | FSRQ: 106
Fold 2/5
  Fit count: 1283 | BLL: 847 | FSRQ: 436
  Held-out count: 321 | BLL: 206 | FSRQ: 115
Fold 3/5
  Fit count: 1283 | BLL: 849 | FSRQ: 434
  Held-out count: 321 | BLL: 204 | FSRQ: 117
Fold 4/5
  Fit count: 1283 | BLL: 833 | FSRQ: 450
  Held-out count: 321 | BLL: 220 | FSRQ: 101
Fold 5/5
  Fit count: 1284 | BLL: 845 | FSRQ: 439
  Held-out count: 320 | BLL: 208 | FSRQ: 112
Saved calibration CSV: agarwal_outputs/agarwal_calibration_mccp.csv
Saved fold summary CSV: agarwal_outputs/agarwal_mccp_fold_summary.csv

Combined calibration metrics:
accuracy              0.913965
precision_weighted    0.913618
recall_weighted       0.913965
f1_weighted           0.913735
auc                   0.963630
dtype: float64



,fold,fit_samples,heldout_samples,fit_bll,fit_fsrq,heldout_bll,heldout_fsrq,best_epoch_by_val_loss,best_val_loss,best_val_accuracy,best_val_auc,best_val_f1_weighted,heldout_accuracy,heldout_auc,heldout_f1_weighted,heldout_csv,best_model_path
0,1,1283,321,838,445,215,106,146,0.238967,0.885,0.958571,0.882840,0.922118,0.961167,0.922024,agarwal_outputs/mccp_heldout_folds/agarwal_hol...,agarwal_outputs/mccp_fold_models/fold_01/final...
1,2,1283,321,847,436,206,115,1058,0.238187,0.905,0.961310,0.903768,0.900312,0.967539,0.899909,agarwal_outputs/mccp_heldout_folds/agarwal_hol...,agarwal_outputs/mccp_fold_models/fold_02/final...
2,3,1283,321,849,434,204,117,478,0.226697,0.900,0.961310,0.898417,0.925234,0.968787,0.924946,agarwal_outputs/mccp_heldout_folds/agarwal_hol...,agarwal_outputs/mccp_fold_models/fold_03/final...
3,4,1283,321,833,450,220,101,157,0.234216,0.880,0.961190,0.877381,0.915888,0.958506,0.915774,agarwal_outputs/mccp_heldout_folds/agarwal_hol...,agarwal_outputs/mccp_fold_models/fold_04/final...
4,5,1284,320,845,439,208,112,103,0.238027,0.895,0.958929,0.893638,0.906250,0.961324,0.906052,agarwal_outputs/mccp_heldout_folds/agarwal_hol...,agarwal_outputs/mccp_fold_models/fold_05/final...


In [6]:
qhat_builder = MCPTool.from_calibration_csv(str(CALIBRATION_CSV), alpha=MCP_ALPHA)
qhat_builder.export_qhat_csv(str(QHAT_CSV))

qhat_df = pd.read_csv(QHAT_CSV)
print("Saved qhat CSV:", QHAT_CSV)
qhat_df.head()


  Saved: agarwal_outputs/agarwal_qhat_mccp.csv
Saved qhat CSV: agarwal_outputs/agarwal_qhat_mccp.csv


,alpha,n_cal,qhat_BLL,qhat_FSRQ
0,0.0100,1604,0.947114,0.981222
1,0.0296,1604,0.730308,0.915127
2,0.0492,1604,0.608633,0.886827
3,0.0688,1604,0.455608,0.810253
4,0.0884,1604,0.358510,0.768103


In [7]:
print("Calibration CSV preview")
display(calibration_df.head())

print("Qhat rows nearest alpha =", MCP_ALPHA)
display(qhat_df.iloc[(qhat_df["alpha"] - MCP_ALPHA).abs().argsort()[:3]].copy())


Calibration CSV preview


,true_label,BLL,FSRQ
0,BLL,0.996505,0.003495
1,BLL,0.144586,0.855414
2,BLL,0.942180,0.057820
3,BLL,0.817485,0.182515
4,BLL,0.999530,0.000470


Qhat rows nearest alpha = 0.1


,alpha,n_cal,qhat_BLL,qhat_FSRQ
5,0.1000,1604,0.313939,0.680488
6,0.1080,1604,0.277245,0.651445
4,0.0884,1604,0.358510,0.768103
